# IOAI — 2024 Summer National Transfer Learning (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('CIFAR-10/100 은 torchvision 으로, 사전학습 ResNet 는 torch.hub 로 자동 다운로드됩니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Transfer Learning (Transfer Tanulás): CIFAR-100 → CIFAR-10

> **HAIO 2024 — Summer National Finals (CV)**

Use a **CIFAR-100-pretrained ResNet** and transfer it to **CIFAR-10**. Score = **CIFAR-10 test accuracy** (canonical torchvision order). **Submit** `submission.csv` (`id,label`).

Baseline: load `cifar100_resnet20` (torch.hub), replace the head (100→10), fine-tune on CIFAR-10.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader
torch.manual_seed(0); device='cuda' if torch.cuda.is_available() else 'cpu'
MEAN,STD=(0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)
tf_tr=T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.ToTensor(),T.Normalize(MEAN,STD)])
tf_te=T.Compose([T.ToTensor(),T.Normalize(MEAN,STD)])
tr=torchvision.datasets.CIFAR10('./data',train=True,download=True,transform=tf_tr)
te=torchvision.datasets.CIFAR10('./data',train=False,download=True,transform=tf_te)
dl_tr=DataLoader(tr,256,shuffle=True,num_workers=2); dl_te=DataLoader(te,512,shuffle=False,num_workers=2)
device

## Load CIFAR-100-pretrained ResNet20, transfer head to 10 classes, fine-tune

In [ ]:
model=torch.hub.load('chenyaofo/pytorch-cifar-models','cifar100_resnet20',pretrained=True)
model.fc=nn.Linear(model.fc.in_features,10)   # transfer: new 10-class head
model=model.to(device)
opt=torch.optim.Adam(model.parameters(),lr=1e-3); crit=nn.CrossEntropyLoss()
@torch.no_grad()
def acc(dl):
    model.eval(); c=t=0
    for x,y in dl:
        p=model(x.to(device)).argmax(1).cpu(); c+=(p==y).sum().item(); t+=y.size(0)
    return c/t
for ep in range(3):
    model.train()
    for x,y in dl_tr:
        x,y=x.to(device),y.to(device); opt.zero_grad(); crit(model(x),y).backward(); opt.step()
    print(f'epoch {ep+1} test_acc {acc(dl_te):.4f}')

## Predict CIFAR-10 test (canonical order) → `submission.csv`

In [ ]:
model.eval(); preds=[]
with torch.no_grad():
    for x,_ in dl_te: preds.append(model(x.to(device)).argmax(1).cpu().numpy())
preds=np.concatenate(preds)
pd.DataFrame({'id':range(len(preds)),'label':preds}).to_csv('submission.csv',index=False); print('wrote',len(preds))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)